# Classifying Caltech101 with Transfer Learning

Assignment from *Programming Exercises Chapter 14*, Stijn Lievens and Simon De Gheselle (June 26, 2026). Each part has its own empty code cell. Add written answers beneath the relevant cell. No solutions are included.

## Setup
Add your imports and setup here.

## 4.1 Load the Dataset

Use `tensorflow_datasets` to load Caltech101. Use all training data for training, the first 50% of the test split for validation, and the last 50% for testing. Keep the data as Datasets rather than converting to NumPy. Dataset: https://www.tensorflow.org/datasets/catalog/caltech101. There are 102 labels: 101 object classes plus background clutter.

Verify dataset sizes: training 3060; validation 3042; test 3042.

Inspect image and label data types using `for image, label in train_ds.take(1):`.

## 4.2 Some Data Exploration

Show the first nine training images in a 3 × 3 grid, with label and image size above each. Notice their different sizes.

Implement `resize(image, height=224, width=224)` using a Keras preprocessing layer. Use Dataset `map` to resize images in `train_ds`, `val_ds`, and `test_ds`, preserving their labels.

Show the images again and verify equal sizes. Adjust the display code for tensors that now have dtype `tf.float32`.

## 4.3 Preprocess the Data

Preprocess images with `keras.applications.mobilenet_v2.preprocess_input`.

Shuffle the training set. Validation and test sets need no shuffling.

Batch all datasets with batch size 32.

### Part 4

Use `prefetch` on the training dataset to prepare the next batch during training.

## 4.4 Create a Data Augmentation Layer

Create a Sequential augmentation model: random horizontal flip; random rotation between -20 and +20 degrees, filling missing pixels with 0 (determine `factor`); random zoom in or out by at most 15%, filling missing pixels with 0.

Check that augmentations look reasonable by showing up to nine augmented versions of the first couple of validation images. You may manually change the selected image index. Datasets are batched; scale pixel values from [-1, 1] to [0, 1] before `plt.imshow`.

## 4.5 Load the Base Model

Load MobileNetV2 with ImageNet weights, without its top layer, for input images of shape (224, 224, 3). Reference: https://keras.io/api/applications/mobilenet/#mobilenetv2-function.

Use the Functional API to compose: data augmentation; MobileNetV2 base; global average pooling; dropout rate 0.4; fully connected output with appropriate units and activation.

Freeze all base weights. Call the base with `training=False` to prevent batch normalization moving-statistic updates. Verify 2,388,646 total parameters, including 130,662 trainable parameters.

## 4.6 Train the Top Layer of the Model

Compile with Adam learning rate 0.001, the correct loss, and accuracy.

Train for at most 100 epochs. Use EarlyStopping with patience 3, monitoring validation accuracy and requiring an improvement of at least 0.1%.

## 4.7 Fine-tune the Model

Find and print base-model layer names. Identify its 16 blocks. Fine-tune the last quarter, from block 13 onward.

Find the index of `block_13_expand`. Set `base_model.trainable = True`, then freeze every layer before that index.

Recompile with Adam learning rate 0.00001 to preserve pretrained weights. Verify 1,812,006 trainable parameters.

### Part 4

Train as before. Validation accuracy may improve only slightly.

## 4.8 Evaluate the Model

Evaluate on the test set. Can you achieve more than 90% accuracy?

## 4.9 Calculate Top-1, Top-3 and Top-5 Accuracy

Use `keras.metrics.SparseTopKCategoricalAccuracy` to measure test top-1, top-3, and top-5 accuracy. The assignment gives achievable results of 90.6%, 97.1%, and 98.6%.

Recompile with the appropriate metrics before calling `evaluate()` again.